# Q1 Appendix: All-metric raw means (Accuracy / Precision / Recall / Macro-F1)

This appendix reproduces the article-ready Q1 tables (`tab_q1_overall`, `tab_q1_per_network`, `tab_q1_per_model`, and the per-condition granularity of `tab_q1_stats`), but **expanded to report all four evaluation metrics** as raw mean-over-runs values, instead of only macro-F1.

The tables are in **tidy / long format**: a single set of metric columns — Accuracy, Precision, Recall, Macro-F1 (mean over runs), in that order — and one added label column **Approach** on the left that carries the values `Component-based` and `Sequence-based` (fixed order: Component-based then Sequence-based). Each source group therefore yields two rows. All headline derived columns (the Δ MacroF1 (Seq−Comp) reference and all statistical machinery — Wilcoxon p, Cohen's d, significance) are dropped.

**Q1 invariants preserved:** drop split "Random with same distribution"; in-network only (train_set == test_set); only the six paired classical models (`knn, lightgbm, logistic-regression, random-forest, svm, xgboost`); `approach = enable_sequences.map({True: 'Sequence-based', False: 'Component-based'})`; average over `run_no`.


## Setup, load, melt, and run-averaging


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)

# ── Paths (notebook runs from A/) ────────────────────────────────────────────
DATA_PATH = Path('../data/wandb_export_final_hyperparameters.csv')
TAB_DIR   = Path('tables'); TAB_DIR.mkdir(exist_ok=True)

NETWORKS = ['SetA', 'SetB', 'SetC', 'SetD']
# All four metrics carried through (NOT filtered to f1_score)
METRICS  = ['f1_score', 'accuracy', 'precision', 'recall']
# Metric column display order: Accuracy, Precision, Recall, Macro-F1
METRIC_ORDER = ['accuracy', 'precision', 'recall', 'f1_score']
METRIC_LABEL = {'accuracy': 'Accuracy', 'precision': 'Precision',
                'recall': 'Recall', 'f1_score': 'Macro-F1'}

# Six paired classical models (in both approaches) — fair comparison
PAIRED_MODELS = ['knn', 'lightgbm', 'logistic-regression',
                 'random-forest', 'svm', 'xgboost']

# ── Load raw data ────────────────────────────────────────────────────────────
raw = pd.read_csv(DATA_PATH, index_col=0)
raw = raw[raw['split'] != 'Random with same distribution']
raw['approach'] = raw['enable_sequences'].map({True: 'Sequence-based',
                                               False: 'Component-based'})

# ── Parse metric columns into long form (ALL metrics) ────────────────────────
metric_cols = [c for c in raw.columns if c.count('/') == 2]
id_vars = ['model', 'task', 'split', 'enable_sequences', 'approach', 'run_no']
long = raw[id_vars + metric_cols].melt(
    id_vars=id_vars, value_vars=metric_cols,
    var_name='metric_key', value_name='value')
long[['train_set', 'test_set', 'metric']] = long['metric_key'].str.split('/', expand=True)
long = long.drop(columns='metric_key')

# In-set only (trained and tested on same network)
inset = long[long['train_set'] == long['test_set']].copy()
inset = inset.rename(columns={'train_set': 'network'}).drop(columns='test_set')

# ── Average across runs (all 4 metrics kept in the `metric` dimension) ───────
group_keys = ['model', 'task', 'split', 'enable_sequences', 'approach',
              'network', 'metric']
avg = (inset.groupby(group_keys, as_index=False)['value']
            .mean().rename(columns={'value': 'mean_value'}))

print(f'Long-form rows (all):    {len(long):,}')
print(f'Long-form rows (in-set): {len(inset):,}')
print(f'Averaged rows (in-set):  {len(avg):,}')
print('Metrics present:', sorted(avg["metric"].unique()))
avg.head()


Long-form rows (all):    38,528
Long-form rows (in-set): 9,632
Averaged rows (in-set):  1,088
Metrics present: ['accuracy', 'f1_score', 'precision', 'recall']


,model,task,split,enable_sequences,approach,network,metric,mean_value
0,gru,Binary,Random split,True,Sequence-based,SetA,accuracy,0.941879
1,gru,Binary,Random split,True,Sequence-based,SetA,f1_score,0.801669
2,gru,Binary,Random split,True,Sequence-based,SetA,precision,0.801336
3,gru,Binary,Random split,True,Sequence-based,SetA,recall,0.807207
4,gru,Binary,Random split,True,Sequence-based,SetB,accuracy,0.993769


The helper below pivots only the `metric` dimension into columns (Accuracy, Precision, Recall, Macro-F1) while keeping `approach` as a row value in an added **Approach** label column. Each source group thus produces two rows — Component-based then Sequence-based (fixed order). No difference or ranking columns are produced.


In [2]:
APPROACH_ORDER = ['Component-based', 'Sequence-based']

def expand_all_metrics(df, index_cols):
    """Tidy/long: pivot only `metric` into columns; keep `approach` as a row.

    Produces one metric column set — Accuracy, Precision, Recall, Macro-F1
    (mean over runs), in that order — plus an added `Approach` label column
    (values Component-based then Sequence-based). Each index group yields two
    rows. No difference or ranking columns are produced.
    """
    g = (df.groupby(index_cols + ['approach', 'metric'], as_index=False)['mean_value']
           .mean())
    piv = g.pivot_table(index=index_cols + ['approach'], columns='metric',
                        values='mean_value')
    piv = piv.reindex(columns=METRIC_ORDER)
    piv.columns = [METRIC_LABEL[m] for m in METRIC_ORDER]
    piv = piv.reset_index()
    # Approach as an ordered categorical so sorting keeps the fixed order
    piv['approach'] = pd.Categorical(piv['approach'], categories=APPROACH_ORDER,
                                     ordered=True)
    piv = piv.sort_values(index_cols + ['approach']).reset_index(drop=True)
    piv = piv.rename(columns={'approach': 'Approach'})
    # Reorder so Approach sits just left of the metric columns
    metric_names = [METRIC_LABEL[m] for m in METRIC_ORDER]
    piv = piv[index_cols + ['Approach'] + metric_names]
    return piv


## q1_appendix_overall

Global comparison by (task, split) — source `tab_q1_overall`. Tidy/long: columns [task, split, Approach, Accuracy, Precision, Recall, Macro-F1]; each (task, split) yields two rows (Component-based then Sequence-based).


In [3]:
data = avg[avg['model'].isin(PAIRED_MODELS)].copy()
t_overall = expand_all_metrics(data, ['task', 'split'])
num_cols = [c for c in t_overall.columns if c not in ('task', 'split', 'Approach')]
t_overall[num_cols] = t_overall[num_cols].round(4)

t_overall.to_csv(TAB_DIR / 'q1_appendix_overall.csv', index=False)
caption = ('Q1 appendix (tidy/long): means over runs of Accuracy, Precision, '
           'Recall, and Macro-F1 for the six paired models, in-network, by task '
           'and split. The Approach label column distinguishes Component-based '
           'from Sequence-based; each (task, split) yields two rows.')
latex = t_overall.to_latex(index=False, float_format='%.4f', escape=False,
                           caption=caption, label='tab:q1_appendix_overall')
(TAB_DIR / 'q1_appendix_overall.tex').write_text(latex)
print('Saved q1_appendix_overall  shape:', t_overall.shape)
t_overall


Saved q1_appendix_overall  shape: (8, 7)


,task,split,Approach,Accuracy,Precision,Recall,Macro-F1
0,Binary,Random split,Component-based,0.8798,0.8376,0.8157,0.8168
1,Binary,Random split,Sequence-based,0.9699,0.9037,0.8786,0.8838
2,Binary,Time split,Component-based,0.8391,0.7754,0.6998,0.7068
3,Binary,Time split,Sequence-based,0.9737,0.8751,0.8053,0.8186
4,Multiclass,Random split,Component-based,0.8825,0.8258,0.7502,0.7667
5,Multiclass,Random split,Sequence-based,0.9727,0.8714,0.7454,0.7793
6,Multiclass,Time split,Component-based,0.8206,0.6877,0.6324,0.6075
7,Multiclass,Time split,Sequence-based,0.9707,0.7989,0.6756,0.7029


## q1_appendix_per_network

Per-network × task comparison — source `tab_q1_per_network`. Tidy/long: columns [network, task, Approach, Accuracy, Precision, Recall, Macro-F1]; each (network, task) yields two rows.


In [4]:
data = avg[avg['model'].isin(PAIRED_MODELS)].copy()
t_net = expand_all_metrics(data, ['network', 'task'])
num_cols = [c for c in t_net.columns if c not in ('network', 'task', 'Approach')]
t_net[num_cols] = t_net[num_cols].round(4)

t_net.to_csv(TAB_DIR / 'q1_appendix_per_network.csv', index=False)
caption = ('Q1 appendix (tidy/long): means over runs of Accuracy, Precision, '
           'Recall, and Macro-F1 for the six paired models, in-network, per '
           'network and task. The Approach label column distinguishes '
           'Component-based from Sequence-based; each (network, task) yields two rows.')
latex = t_net.to_latex(index=False, float_format='%.4f', escape=False,
                       caption=caption, label='tab:q1_appendix_per_network')
(TAB_DIR / 'q1_appendix_per_network.tex').write_text(latex)
print('Saved q1_appendix_per_network  shape:', t_net.shape)
t_net


Saved q1_appendix_per_network  shape: (16, 7)


,network,task,Approach,Accuracy,Precision,Recall,Macro-F1
0,SetA,Binary,Component-based,0.8638,0.7782,0.7233,0.7403
1,SetA,Binary,Sequence-based,0.9267,0.7031,0.6407,0.6343
2,SetA,Multiclass,Component-based,0.8527,0.6312,0.6668,0.6193
3,SetA,Multiclass,Sequence-based,0.9371,0.6608,0.5182,0.5450
4,SetB,Binary,Component-based,0.8215,0.7427,0.6314,0.6308
5,SetB,Binary,Sequence-based,0.9907,0.9431,0.9204,0.9265
6,SetB,Multiclass,Component-based,0.8345,0.7857,0.6437,0.6562
7,SetB,Multiclass,Sequence-based,0.9902,0.9114,0.8032,0.8382
8,SetC,Binary,Component-based,0.7712,0.7233,0.6966,0.6955
9,SetC,Binary,Sequence-based,0.9872,0.9370,0.8796,0.8981


## q1_appendix_per_model

Per-model comparison — source `tab_q1_per_model`. Tidy/long: columns [model, Approach, Accuracy, Precision, Recall, Macro-F1]. Models are sorted by the model's overall mean Macro-F1 (averaged across approaches) descending, then Approach in fixed order (Component-based, Sequence-based).


In [5]:
data = avg[avg['model'].isin(PAIRED_MODELS)].copy()
t_model = expand_all_metrics(data, ['model'])

# Sort by each model's overall mean Macro-F1 (across approaches) DESCENDING,
# then Approach in fixed order.
model_rank = (t_model.groupby('model')['Macro-F1'].mean()
                     .sort_values(ascending=False))
t_model['model'] = pd.Categorical(t_model['model'],
                                  categories=list(model_rank.index), ordered=True)
t_model = t_model.sort_values(['model', 'Approach']).reset_index(drop=True)
t_model['model'] = t_model['model'].astype(str)

num_cols = [c for c in t_model.columns if c not in ('model', 'Approach')]
t_model[num_cols] = t_model[num_cols].round(4)

t_model.to_csv(TAB_DIR / 'q1_appendix_per_model.csv', index=False)
caption = ('Q1 appendix (tidy/long): means over runs of Accuracy, Precision, '
           'Recall, and Macro-F1 for the six paired models, in-network (averaged '
           'over networks, tasks, and splits), per model. The Approach label '
           'column distinguishes Component-based from Sequence-based; models are '
           'ordered by overall mean Macro-F1 across approaches (descending).')
latex = t_model.to_latex(index=False, float_format='%.4f', escape=False,
                         caption=caption, label='tab:q1_appendix_per_model')
(TAB_DIR / 'q1_appendix_per_model.tex').write_text(latex)
print('Saved q1_appendix_per_model  shape:', t_model.shape)
t_model


Saved q1_appendix_per_model  shape: (12, 6)


,model,Approach,Accuracy,Precision,Recall,Macro-F1
0,lightgbm,Component-based,0.8655,0.8074,0.7456,0.7493
1,lightgbm,Sequence-based,0.9782,0.8983,0.8596,0.8717
2,xgboost,Component-based,0.8637,0.8084,0.7449,0.7478
3,xgboost,Sequence-based,0.9818,0.9182,0.8495,0.8698
4,random-forest,Component-based,0.8576,0.7883,0.7470,0.7493
5,random-forest,Sequence-based,0.9828,0.9137,0.8431,0.8650
6,knn,Component-based,0.8519,0.7781,0.7335,0.7378
7,knn,Sequence-based,0.9662,0.8404,0.7529,0.7833
8,logistic-regression,Component-based,0.8469,0.7413,0.6696,0.6697
9,logistic-regression,Sequence-based,0.9678,0.8156,0.6971,0.7239


## q1_appendix_per_condition

Raw-means replacement for the statistics table — source `tab_q1_stats` granularity. Tidy/long: columns [network, task, split, Approach, Accuracy, Precision, Recall, Macro-F1]. All statistical machinery (Wilcoxon p, Cohen's d, significance) and difference columns are dropped; only raw means remain, with the Approach label carrying Component-based / Sequence-based.


In [6]:
data = avg[avg['model'].isin(PAIRED_MODELS)].copy()
t_cond = expand_all_metrics(data, ['network', 'task', 'split'])
num_cols = [c for c in t_cond.columns if c not in ('network', 'task', 'split', 'Approach')]
t_cond[num_cols] = t_cond[num_cols].round(4)

t_cond.to_csv(TAB_DIR / 'q1_appendix_per_condition.csv', index=False)
caption = ('Q1 appendix (tidy/long): means over runs of Accuracy, Precision, '
           'Recall, and Macro-F1 for the six paired models, in-network, per '
           'network, task, and split. The Approach label column distinguishes '
           'Component-based from Sequence-based; each (network, task, split) '
           'yields two rows.')
latex = t_cond.to_latex(index=False, float_format='%.4f', escape=False,
                        caption=caption, label='tab:q1_appendix_per_condition')
(TAB_DIR / 'q1_appendix_per_condition.tex').write_text(latex)
print('Saved q1_appendix_per_condition  shape:', t_cond.shape)
t_cond


Saved q1_appendix_per_condition  shape: (32, 8)


,network,task,split,Approach,Accuracy,Precision,Recall,Macro-F1
0,SetA,Binary,Random split,Component-based,0.9064,0.8882,0.8415,0.8611
1,SetA,Binary,Random split,Sequence-based,0.9175,0.7610,0.7266,0.7285
2,SetA,Binary,Time split,Component-based,0.8212,0.6683,0.6051,0.6196
3,SetA,Binary,Time split,Sequence-based,0.9358,0.6452,0.5549,0.5401
4,SetA,Multiclass,Random split,Component-based,0.8941,0.7987,0.7250,0.7458
5,SetA,Multiclass,Random split,Sequence-based,0.9396,0.7429,0.6103,0.6416
6,SetA,Multiclass,Time split,Component-based,0.8113,0.4637,0.6087,0.4927
7,SetA,Multiclass,Time split,Sequence-based,0.9346,0.5787,0.4262,0.4484
8,SetB,Binary,Random split,Component-based,0.8263,0.7231,0.6825,0.6743
9,SetB,Binary,Random split,Sequence-based,0.9911,0.9505,0.9446,0.9451
